# AAI612: Deep Learning & its Applications

*Final Project: Seeing the Market, Predicting Stock Price Direction from Chart Images*

<a href="https://colab.research.google.com/github/Kablox/AAI612-Final-Project/blob/main/AAI612_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

Traders look at price charts to decide when to buy or sell. In this project a ConvNet looks at the same kind of chart. Every 20 trading days of a stock are drawn as a small black and white chart, and the model has to predict if the price will be higher or lower 5 trading days later. The chart format follows Jiang, Kelly and Xiu, "(Re-)Imag(in)ing Price Trends", The Journal of Finance, 2023.

The ConvNet is compared with simple baselines, an LSTM that reads the same data as numbers, and a pretrained ResNet50.

In [ ]:
import ssl
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

ssl._create_default_https_context = ssl._create_unverified_context

## Part 1: Loading the Data

Daily prices (open, high, low, close and volume) of 50 large US companies from December 2009 to December 2024, downloaded once from Yahoo Finance with the `yfinance` library and saved as a CSV file. The prices are adjusted for stock splits, so a split doesn't look like a crash in the charts.

In [ ]:
data_path = 'https://raw.githubusercontent.com/Kablox/AAI612-Final-Project/main/data/stock_prices.csv'
df = pd.read_csv(data_path)

print('Rows: %d, stocks: %d' % (len(df), df['Ticker'].nunique()))
print('From %s to %s' % (df['Date'].min(), df['Date'].max()))
df.head()

### Looking at one stock

In [ ]:
apple = df[df['Ticker'] == 'AAPL']
plt.figure(figsize=(10, 4))
plt.plot(pd.to_datetime(apple['Date']), apple['Close'])
plt.title('Apple closing price')
plt.xlabel('date')
plt.ylabel('price ($)')
plt.show()

## Part 2: Building the Samples

Each sample is one window of 20 trading days of one stock. The label is 1 if the closing price 5 trading days after the window is higher than the last closing price of the window, and 0 otherwise. The windows move forward 5 days at a time.

Every window is saved in two forms: a chart image for the ConvNet, and the daily numbers (returns and volume) for the LSTM and the logistic regression.

In [ ]:
window_days = 20     # days in each chart
horizon = 5          # predict the price 5 trading days later
stride = 5           # move the window 5 days at a time
image_height = 64
image_width = 3 * window_days
price_height = 51    # top part of the image for prices, the rest for volume

### Drawing a chart

Each day takes 3 pixels: a tick on the left for the opening price, a bar from the low to the high in the middle, and a tick on the right for the closing price. A 20-day moving average line goes through the middle of each day, and the volume bars are at the bottom.

In [ ]:
def draw_chart(open_values, high_values, low_values, close_values, ma_values, volume_values):
    ''' Draws one 20-day window as a 64x60 black and white chart '''
    image = np.zeros((image_height, image_width), dtype=np.uint8)

    # scale the prices so the window fills the price area, row 0 is the top
    lowest = min(low_values.min(), ma_values.min())
    highest = max(high_values.max(), ma_values.max())
    def price_row(price):
        return (price_height - 1 - np.round((price - lowest) / (highest - lowest) * (price_height - 1))).astype(int)
    open_rows = price_row(open_values)
    high_rows = price_row(high_values)
    low_rows = price_row(low_values)
    close_rows = price_row(close_values)
    ma_rows = price_row(ma_values)

    # volume bars, scaled by the biggest volume in the window
    volume_heights = np.round(volume_values / volume_values.max() * (image_height - price_height - 1)).astype(int)

    for day in range(window_days):
        x = 3 * day
        image[open_rows[day], x] = 1
        image[high_rows[day]:low_rows[day] + 1, x + 1] = 1
        image[close_rows[day], x + 2] = 1
        if volume_heights[day] > 0:
            image[image_height - volume_heights[day]:, x + 1] = 1

    # moving average line connecting the middle pixel of each day
    ma_x = 3 * np.arange(window_days) + 1
    line_x = np.arange(ma_x[0], ma_x[-1] + 1)
    line_rows = np.round(np.interp(line_x, ma_x, ma_rows)).astype(int)
    image[line_rows, line_x] = 1
    return image

### Building all the windows

For the numbers version, each day of a window has 5 values: the daily return, the open, high and low compared to the close, and the volume compared to the average volume of the window.

In [ ]:
images, sequences, labels = [], [], []
past_returns, future_returns, end_dates, future_dates, tickers = [], [], [], [], []

start_time = time.time()
for ticker, stock in df.groupby('Ticker'):
    stock = stock.sort_values('Date').reset_index(drop=True)
    stock['MA'] = stock['Close'].rolling(window_days).mean()
    open_values = stock['Open'].values
    high_values = stock['High'].values
    low_values = stock['Low'].values
    close_values = stock['Close'].values
    volume_values = stock['Volume'].values
    ma_values = stock['MA'].values
    dates = stock['Date'].values

    # start at day 19 so the moving average is available for every day of the window
    for start in range(window_days - 1, len(stock) - window_days - horizon + 1, stride):
        end = start + window_days - 1
        future = end + horizon
        days = slice(start, end + 1)

        images.append(draw_chart(open_values[days], high_values[days], low_values[days],
                                 close_values[days], ma_values[days], volume_values[days]))

        daily_returns = close_values[days] / close_values[start - 1:end] - 1
        sequences.append(np.stack([daily_returns,
                                   open_values[days] / close_values[days] - 1,
                                   high_values[days] / close_values[days] - 1,
                                   low_values[days] / close_values[days] - 1,
                                   volume_values[days] / volume_values[days].mean()], axis=1))

        labels.append(int(close_values[future] > close_values[end]))
        past_returns.append(close_values[end] / close_values[end - horizon] - 1)
        future_returns.append(close_values[future] / close_values[end] - 1)
        end_dates.append(dates[end])
        future_dates.append(dates[future])
        tickers.append(ticker)

images = np.array(images)
sequences = np.array(sequences, dtype=np.float32)
labels = np.array(labels)
past_returns = np.array(past_returns)
future_returns = np.array(future_returns)
end_dates = np.array(end_dates)
future_dates = np.array(future_dates)
tickers = np.array(tickers)

print('Built %d windows in %.1f s' % (len(images), time.time() - start_time))
print('Images: %s, sequences: %s' % (str(images.shape), str(sequences.shape)))

### Splitting by time

The data is split by time and not randomly, so the model is always tested on years it has never seen. A window is dropped if its label comes from the next period, otherwise the model would peek into the future.

In [ ]:
def period(dates):
    ''' train for 2010 to 2019, val for 2020 to 2021, test for 2022 to 2024 '''
    years = np.array([int(date[:4]) for date in dates])
    return np.where(years <= 2019, 'train', np.where(years <= 2021, 'val', 'test'))

end_period = period(end_dates)
keep = end_period == period(future_dates)
print('Dropped %d windows at the split boundaries' % (~keep).sum())

train = keep & (end_period == 'train')
val = keep & (end_period == 'val')
test = keep & (end_period == 'test')

for name, split in [('Training', train), ('Validation', val), ('Test', test)]:
    print('%-10s %5d windows, %.1f%% went up' % (name, split.sum(), labels[split].mean() * 100))

### Looking at some charts

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(8):
    index = i * 4000
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[index], cmap='gray')
    plt.title('%s %s, %s' % (tickers[index], end_dates[index], 'up' if labels[index] == 1 else 'down'), fontsize=9)
    plt.axis('off')
plt.show()

## Part 3: Baselines

Before any deep learning, three simple ways to guess. As we saw with the temperature forecasting in Notebook 6.2, a deep model is only useful if it beats these.

* Always up: predict up for every window, since stocks go up a bit more often than down.
* Momentum: predict the same direction as the last 5 days.
* Logistic regression on the 20 daily returns of the window.

Besides accuracy, the AUC is used. It measures how well a model ranks the windows from most likely up to least likely up. 0.5 is a coin flip and 1.0 is perfect.

In [ ]:
results = []

def report(name, split_name, y_true, y_score):
    ''' Prints accuracy and AUC, y_score is the predicted probability of going up '''
    accuracy = np.mean((y_score > 0.5) == y_true)
    auc = roc_auc_score(y_true, y_score)
    print('%-20s %-10s accuracy %.2f%%, AUC %.3f' % (name, split_name, accuracy * 100, auc))
    if split_name == 'test':
        results.append((name, accuracy, auc))

for split_name, split in [('validation', val), ('test', test)]:
    # always up gives everything the same score, so its AUC is 0.5
    report('Always up', split_name, labels[split], np.full(split.sum(), 0.51))
    # momentum: up if the last 5 days went up, the 5-day return is used as the score
    report('Momentum', split_name, labels[split], 0.5 + past_returns[split])

In [ ]:
# logistic regression on the 20 daily returns, standardized with the training data only
returns_train = sequences[train][:, :, 0]
returns_mean = returns_train.mean(axis=0)
returns_std = returns_train.std(axis=0)

logistic = LogisticRegression(max_iter=1000)
logistic.fit((returns_train - returns_mean) / returns_std, labels[train])

for split_name, split in [('validation', val), ('test', test)]:
    scores = logistic.predict_proba((sequences[split][:, :, 0] - returns_mean) / returns_std)[:, 1]
    report('Logistic regression', split_name, labels[split], scores)

## Part 4: LSTM on the Numbers

The LSTM reads the 20 days of each window in order, 5 numbers per day. The numbers are standardized with the mean and standard deviation of the training windows only, the lesson from Notebook 6.3.

Every deep model is trained 3 times with different seeds and the 3 predictions are averaged, so one lucky or unlucky run doesn't decide the result. Early stopping keeps the epoch with the lowest validation loss.

In [ ]:
epochs = 20
seeds = [0, 1, 2]
batch_size = 128

In [ ]:
def train_model(build_model, x_train, x_val, seed):
    ''' Trains one model with early stopping on the validation loss '''
    tf.keras.utils.set_random_seed(seed)
    model = build_model()
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    history = model.fit(x_train, labels[train], validation_data=(x_val, labels[val]),
                        epochs=epochs, batch_size=batch_size, callbacks=[early_stopping], verbose=2)
    return model, history

def train_seeds(name, build_model, x_train, x_val, x_test):
    ''' Trains the model once per seed and averages the predictions of all seeds '''
    models, histories, val_scores, test_scores = [], [], [], []
    start_time = time.time()
    for seed in seeds:
        model, history = train_model(build_model, x_train, x_val, seed)
        val_scores.append(model.predict(x_val, verbose=0)[:, 0])
        test_scores.append(model.predict(x_test, verbose=0)[:, 0])
        print('%s seed %d: %d epochs, test accuracy %.2f%%' % (name, seed, len(history.history['loss']),
              np.mean((test_scores[-1] > 0.5) == labels[test]) * 100))
        models.append(model)
        histories.append(history)
    print('%s trained in %.1f minutes' % (name, (time.time() - start_time) / 60))

    val_score = np.mean(val_scores, axis=0)
    test_score = np.mean(test_scores, axis=0)
    report(name, 'validation', labels[val], val_score)
    report(name, 'test', labels[test], test_score)
    return models, histories, val_score, test_score

In [ ]:
sequence_mean = sequences[train].mean(axis=(0, 1))
sequence_std = sequences[train].std(axis=(0, 1))
x_train_seq = (sequences[train] - sequence_mean) / sequence_std
x_val_seq = (sequences[val] - sequence_mean) / sequence_std
x_test_seq = (sequences[test] - sequence_mean) / sequence_std

def build_lstm():
    ''' LSTM with 32 units reading the 20 days in order '''
    return tf.keras.models.Sequential([keras.layers.Input(shape=(window_days, 5)),
                                       keras.layers.LSTM(32),
                                       keras.layers.Dropout(0.5),
                                       keras.layers.Dense(1, activation='sigmoid')])

lstm_models, lstm_histories, lstm_val, lstm_test = train_seeds('LSTM', build_lstm, x_train_seq, x_val_seq, x_test_seq)

## Part 5: ConvNet on the Chart Images

The ConvNet has 3 blocks of 5x3 convolutions. The pooling only shrinks the height (2x1), like in the paper, so every day keeps its own columns until the end.

Two sizes are trained: a small one (32, 64 and 128 filters) and the paper size (64, 128 and 256 filters), to see if the bigger network is worth the extra time.

In [ ]:
x_train_img = images[train][..., np.newaxis].astype('float32')
x_val_img = images[val][..., np.newaxis].astype('float32')
x_test_img = images[test][..., np.newaxis].astype('float32')

def build_convnet(filters):
    ''' ConvNet with 3 blocks of 5x3 convolutions and 2x1 max pooling '''
    model = tf.keras.models.Sequential([keras.layers.Input(shape=(image_height, image_width, 1))])
    for i in range(len(filters)):
        model.add(keras.layers.Conv2D(filters[i], (5, 3), padding='same', activation='relu', name='conv%d' % (i + 1)))
        model.add(keras.layers.MaxPooling2D((2, 1)))
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dropout(0.5))
    model.add(keras.layers.Dense(1, activation='sigmoid'))
    return model

build_convnet([32, 64, 128]).summary()

In [ ]:
small_convnet = train_seeds('ConvNet small', lambda: build_convnet([32, 64, 128]), x_train_img, x_val_img, x_test_img)

In [ ]:
paper_convnet = train_seeds('ConvNet paper size', lambda: build_convnet([64, 128, 256]), x_train_img, x_val_img, x_test_img)

## Part 6: Transfer Learning with ResNet50

Like VGG16 in Notebook 4.4, a ResNet50 trained on ImageNet is frozen and only a small classifier on top is trained. Since the ResNet50 is frozen, every chart only has to go through it once, so its 2048 features are computed once and saved. The charts are resized to 64x64 and copied into 3 color channels, because ResNet50 expects color images.

In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(64, 64, 3))
base_model.trainable = False

def prepare_for_resnet(batch):
    ''' Turns a batch of 0/1 charts into 64x64 color images the way ResNet50 expects '''
    batch = tf.cast(batch, tf.float32)[..., tf.newaxis] * 255.0
    batch = tf.image.resize(batch, (64, 64))
    batch = tf.image.grayscale_to_rgb(batch)
    return preprocess_input(batch)

def resnet_features(split):
    ''' Runs the charts through the frozen ResNet50 once and returns 2048 features per chart '''
    dataset = tf.data.Dataset.from_tensor_slices(images[split]).batch(256).map(prepare_for_resnet)
    return base_model.predict(dataset, verbose=0)

start_time = time.time()
features_train = resnet_features(train)
features_val = resnet_features(val)
features_test = resnet_features(test)
print('ResNet50 features: %s in %.1f minutes' % (str(features_train.shape), (time.time() - start_time) / 60))

In [ ]:
def build_resnet_head():
    ''' Small classifier on top of the 2048 ResNet50 features '''
    return tf.keras.models.Sequential([keras.layers.Input(shape=(2048,)),
                                       keras.layers.Dense(64, activation='relu'),
                                       keras.layers.Dropout(0.5),
                                       keras.layers.Dense(1, activation='sigmoid')])

resnet = train_seeds('ResNet50 (frozen)', build_resnet_head, features_train, features_val, features_test)

## Part 7: Results

In [ ]:
results_table = pd.DataFrame(results, columns=['Model', 'Test accuracy (%)', 'Test AUC'])
results_table['Test accuracy (%)'] = (results_table['Test accuracy (%)'] * 100).round(2)
results_table['Test AUC'] = results_table['Test AUC'].round(3)
results_table

With 7,500 test windows, a small difference in accuracy can happen just by chance. The cell below computes how big a difference has to be before we can trust it (95% confidence).

In [ ]:
up_share = labels[test].mean()
margin = 1.96 * np.sqrt(up_share * (1 - up_share) / test.sum())
print('Differences smaller than about %.2f percentage points can happen by chance' % (margin * 100))

## Additional Features

The rest of the notebook looks closer at the best ConvNet. The best one is picked using the validation years, not the test years, so the test set stays untouched.

In [ ]:
convnets = {'ConvNet small': small_convnet, 'ConvNet paper size': paper_convnet}
best_name = max(convnets, key=lambda name: roc_auc_score(labels[val], convnets[name][2]))
best_models, best_histories, best_val, best_test = convnets[best_name]
print('Best ConvNet on the validation years: %s' % best_name)

### Learning Curves

Using `plot_learning()` from Notebook 3.2. The red line is the training loss and the blue line is the loss on the validation years.

In [ ]:
def plot_learning():
    plt.plot(chart_x, chart_y_train, 'r-',label='training error')
    plt.plot(chart_x, chart_y_test, 'b-',
    label='test error')
    #plt.axis([0, len(chart_x), 0.0, 1.0])
    plt.xlabel('training epochs')
    plt.ylabel('error')
    plt.legend()
    plt.show()

history = best_histories[0]
chart_x = range(1, len(history.history['loss']) + 1)
chart_y_train = history.history['loss']
chart_y_test = history.history['val_loss']
plot_learning()

### Confusion Matrix

Like in Notebook 3.5, to see if the model really separates up and down weeks or just says up most of the time.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels[test], best_test > 0.5)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['down', 'up'], yticklabels=['down', 'up'])
plt.xlabel('predicted')
plt.ylabel('true')
plt.show()
print('The model predicted up for %.1f%% of the test windows' % ((best_test > 0.5).mean() * 100))

### Grad-CAM: Where Does the ConvNet Look?

Grad-CAM uses the gradients of the last convolution layer to show which parts of a chart pushed the prediction towards up. Because the pooling only shrinks the height, every 3 columns of the heatmap are still one day, so the heatmaps can be averaged per day.

In [ ]:
def grad_cam(model, image_batch):
    ''' Returns one Grad-CAM heatmap per chart (8 rows x 60 columns) '''
    with tf.GradientTape() as tape:
        # run the layers one by one and keep the output of the last convolution
        x = tf.convert_to_tensor(image_batch)
        for layer in model.layers:
            x = layer(x)
            if layer.name == 'conv3':
                conv_output = x
                tape.watch(conv_output)
        prediction = x
    gradients = tape.gradient(prediction, conv_output)
    filter_weights = tf.reduce_mean(gradients, axis=(1, 2))
    heatmaps = tf.nn.relu(tf.reduce_sum(conv_output * filter_weights[:, tf.newaxis, tf.newaxis, :], axis=-1))
    return heatmaps.numpy()

cam_images = x_test_img[:1000]
heatmaps = np.concatenate([grad_cam(best_models[0], cam_images[i:i + 250]) for i in range(0, len(cam_images), 250)])
heatmaps = heatmaps / (heatmaps.max(axis=(1, 2), keepdims=True) + 1e-8)

# every 3 columns is one day, so average them per day
day_importance = heatmaps.mean(axis=(0, 1)).reshape(window_days, 3).mean(axis=1)
plt.figure(figsize=(8, 3))
plt.bar(range(1, window_days + 1), day_importance)
plt.xticks(range(1, window_days + 1))
plt.xlabel('day of the window (20 = most recent)')
plt.ylabel('average Grad-CAM')
plt.title('Which days the ConvNet looks at')
plt.show()

In [ ]:
# the 4 charts the model is most sure will go up, with their heatmap on top
most_sure = np.argsort(best_test[:1000])[-4:]
plt.figure(figsize=(12, 3))
for i in range(4):
    index = most_sure[i]
    plt.subplot(1, 4, i + 1)
    plt.imshow(cam_images[index, :, :, 0], cmap='gray')
    plt.imshow(np.repeat(heatmaps[index], image_height // heatmaps.shape[1], axis=0), cmap='jet', alpha=0.4)
    plt.title('%s %s, P(up) %.2f' % (tickers[test][index], end_dates[test][index], best_test[index]), fontsize=9)
    plt.axis('off')
plt.show()

### Trading Test

Every week of the test years, "buy" the 10 stocks the model is most sure will go up and hold them for 5 days. This is compared to the 10 stocks it likes least and to buying all 50 equally. Trading fees are ignored, so this only shows if the predictions point in a useful direction.

In [ ]:
test_table = pd.DataFrame({'date': end_dates[test], 'ticker': tickers[test],
                           'score': best_test, 'future_return': future_returns[test]})
weekly = []
for date, week in test_table.groupby('date'):
    ranked = week.sort_values('score', ascending=False)['future_return'].values
    weekly.append((date, ranked[:10].mean(), ranked[-10:].mean(), ranked.mean()))
weekly = pd.DataFrame(weekly, columns=['date', 'top 10', 'bottom 10', 'all 50'])

for column in ['top 10', 'bottom 10', 'all 50']:
    print('%-10s average weekly return %.3f%%, $1 grows to $%.2f' % (column, weekly[column].mean() * 100,
          (1 + weekly[column]).prod()))

plt.figure(figsize=(10, 4))
for column in ['top 10', 'bottom 10', 'all 50']:
    plt.plot(pd.to_datetime(weekly['date']), (1 + weekly[column]).cumprod(), label=column)
plt.xlabel('date')
plt.ylabel('value of $1')
plt.legend()
plt.show()

## Observations

To be written after the full run on Colab.